# CPU scoring recovery from exact Kaggle Output version 89

Select Accelerator None, enable Internet, enable Kaggle API Secret codex with access to ducanh3/notebook7bc83c4c76/89, and Save & Run All. Only the six verified scoring files are requested. No Add Input needed. Send csa_scst_xe2500_cpu_recovery_v89_review.zip.


In [ ]:
EXPECTED={'captions_rank0.json': '028a0474148dc3aaa051c5a9002e0147d46e66e7cbdee94cf757f413a754ffe3', 'captions_rank1.json': 'ba3eadc14dc46dc5da34045da4e28790611a53329565665aa4d63f0fb20a6f06', 'test_references.json': '219cbe174a62e04b75c4f110242e12c51edb32a4c42210713db93b330340e75e', 'pilot_checkpoint_receipt.json': 'af1a8dd7141852260ebc25080ce73a19f50129b484f542c442010c6da2d44964', 'worker0.json': 'e5fe96f8a98ee1e890d84f953103f1f018461ad5fd75fffcd991ba8b2b26e040', 'worker1.json': '74022eadc0ec2f27149453d0a061bcec494925573a078dc4d640e1c532521e86'}
CHECKPOINT={'sha256': 'd58f232ad2fb4d829a152610c08bb4ffb84ed2ecb3edbd1a0a85169bf15f3e2c', 'updates': 2500, 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef'}
REFERENCE={'CIDEr': 1.202954787733369, 'CHAIRs': 0.04, 'CHAIRi': 0.027915291529152914, 'Recall': 0.4269839276745354}

from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile, urllib.request

SOURCE_VERSION='ducanh3/notebook7bc83c4c76/89'
SECRET_NAME='codex'
OUTPUT=Path('/kaggle/working/csa_scst_xe2500_cpu_recovery_v89')
def sha(path):
    with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def read(path):return json.loads(path.read_text(encoding='utf8'))
def save(path,value):path.write_text(json.dumps(value,indent=2,ensure_ascii=False,allow_nan=False)+'\n',encoding='utf8')
def download(url,path,digest):
    with urllib.request.urlopen(url,timeout=180) as src,path.open('wb') as dst:
        while chunk:=src.read(1024*1024):dst.write(chunk)
    if sha(path)!=digest:raise ValueError('Changed asset '+path.name)

import json,os,re,subprocess,sys,tempfile,shutil
from kaggle_secrets import UserSecretsClient
try:token=UserSecretsClient().get_secret(SECRET_NAME).strip()
except Exception:raise RuntimeError('Enable the named Secret for the owner of Kaggle Output version 89') from None
if not token or token.startswith('sk-'):raise RuntimeError('Use a Kaggle API token in the named Secret')
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(k,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    process=subprocess.Popen(list(map(str,args)),env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace')
    for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
    if process.wait():raise RuntimeError('API command failed; preserve staged files and read redacted log above')
run_cli([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],authenticated=False)
run_cli([sys.executable,'-u','-c',"import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as folder, patch.object(api,'build_kaggle_client') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output('ducanh3/notebook7bc83c4c76/89',path=folder)\n    req=endpoint.call_args.args[0]\n    assert (req.user_name,req.kernel_slug,req.version_label)==('ducanh3','notebook7bc83c4c76','v89')\n"],authenticated=False)

download_root=Path(tempfile.mkdtemp(prefix='scst2500-v89-',dir='/tmp'))
cli=[sys.executable,'-u','-c','from kaggle.cli import main; main()']
pattern=r'(^|/)(?:'+'|'.join(re.escape(name) for name in EXPECTED)+r')$'
run_cli([*cli,'kernels','output',SOURCE_VERSION,'-p',download_root,'--file-pattern',pattern])
matches=[p.parent for p in download_root.rglob('pilot_checkpoint_receipt.json')
         if p.is_file() and sha(p)==EXPECTED[p.name]]
if len(matches)!=1:raise ValueError('Exact version 89 did not return the expected scoring receipt')
root=matches[0]
for name,digest in EXPECTED.items():
    if not (root/name).is_file() or sha(root/name)!=digest:raise ValueError('Missing/changed saved Output file: '+name)
refs=read(root/'test_references.json')
rows=sorted(read(root/'captions_rank0.json')+read(root/'captions_rank1.json'),key=lambda r:r['coco_id'])
if len(refs)!=5000 or len(rows)!=5000 or len({r['coco_id'] for r in rows})!=5000 or {r['coco_id'] for r in rows}!={r['coco_id'] for r in refs}:
    raise ValueError('Wrong VALID5000 references or captions')
OUTPUT.mkdir()
deps=OUTPUT/'deps'; assets=OUTPUT/'assets';assets.mkdir()
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--target',str(deps),'pycocoevalcap==1.2','nltk==3.9.2','future','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],check=True)
sys.path.insert(0,str(deps))
os.environ['PYTHONPATH']=str(deps)+os.pathsep+os.environ.get('PYTHONPATH','')
base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
download(base+'chair.py',assets/'chair.py','98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d')
download(base+'chair.pkl',assets/'chair.pkl','73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3')
download('https://github.com/adoptium/temurin8-binaries/releases/download/jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz',assets/'java8.tar.gz','6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e')
with tarfile.open(assets/'java8.tar.gz') as archive:archive.extractall(assets/'jre',filter='data')
java,=(assets/'jre').rglob('bin/java')
os.environ['PATH']=str(java.parent)+os.pathsep+os.environ['PATH']
os.environ['JAVA_HOME']=str(java.parent.parent)
subprocess.run([sys.executable,'-c',"import nltk;assert nltk.download('punkt');assert nltk.download('punkt_tab')"],check=True)

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
tok=PTBTokenizer()
gts=tok.tokenize({r['coco_id']:[dict(caption=c) for c in r['references']] for r in refs})
hyp=tok.tokenize({r['coco_id']:[dict(caption=r['caption'])] for r in rows})
if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError('Incomplete tokenization')
cid,per=Cider().compute_score(gts,hyp)
bleu,_=Bleu(4).compute_score(gts,hyp)
rouge,_=Rouge().compute_score(gts,hyp)
save(OUTPUT/'cider_per_image.json',[dict(coco_id=i,CIDEr=float(v)) for i,v in zip(gts,per,strict=True)])
save(OUTPUT/'chair_input.json',[dict(image_id=r['coco_id'],caption=r['caption']) for r in rows])
subprocess.run([sys.executable,'-u',str(assets/'chair.py'),'--cap_file',str(OUTPUT/'chair_input.json'),'--cache',str(assets/'chair.pkl'),'--save_path',str(OUTPUT/'chair_output.json')],cwd=assets,check=True,timeout=600)
chair=read(OUTPUT/'chair_output.json')
sentences=chair['sentences'];captions={r['coco_id']:r['caption'] for r in rows}
if len(sentences)!=5000 or {r['image_id'] for r in sentences}!=set(captions) or any(r['caption']!=captions[r['image_id']] for r in sentences):
    raise ValueError('Wrong CHAIR caption coverage')
workers=[read(root/f'worker{i}.json') for i in range(2)]
metrics=dict(CIDEr=float(cid),BLEU_1=float(bleu[0]),BLEU_4=float(bleu[3]),ROUGE_L=float(rouge),
             NLL=sum(w['NLL_sum'] for w in workers)/sum(w['NLL_tokens'] for w in workers))
metrics.update({k:chair['overall_metrics'][k] for k in ('CHAIRs','CHAIRi','Recall')})
gates=dict(CIDEr=metrics['CIDEr']>=REFERENCE['CIDEr']+.01,
           CHAIRs=metrics['CHAIRs']<=REFERENCE['CHAIRs']+.002,
           Recall=metrics['Recall']>=REFERENCE['Recall']-.005)
result=dict(status='CSA_SCST_XE2500_VALID_COMPLETE',updates=2500,images=5000,split='val',test_used=False,reranking=False,
            metrics=metrics,reference=REFERENCE,delta={k:metrics[k]-v for k,v in REFERENCE.items()},
            checkpoint=CHECKPOINT,gates=gates,gate_passed=all(gates.values()),recovered_from_saved_captions=True)
save(OUTPUT/'result.json',result)
import zipfile
review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ('result.json','cider_per_image.json','chair_output.json'):
        z.write(OUTPUT/name,name)
print(json.dumps(result,indent=2))
print('SEND REVIEW:',review)
